In [ ]:
!pip install vaderSentiment -q

import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

try:
    from google.colab import files
    print('Upload your two labelled CSV files:')
    print('  - Sheet 1-Table_1.csv')
    print('  - Sheet 2-Table_1.csv')
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')
except ImportError:
    print('Not in Colab — reading from local directory.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00
Upload your two labelled CSV files:
  - Sheet 1-Table_1.csv
  - Sheet 2-Table_1.csv


Saving Sheet 1-Table 1.csv to Sheet 1-Table 1.csv
Saving Sheet 2-Table 1.csv to Sheet 2-Table 1.csv
Uploaded: ['Sheet 1-Table 1.csv', 'Sheet 2-Table 1.csv']


In [ ]:
# ---------------------------------------------------------------
# CANONICAL CATEGORY MAPPING
# ---------------------------------------------------------------
CATEGORY_MAP = {
    # DIMENSION 1 — Match Quality & Relationship Intentionality
    'Good match':                    'Good quality matches',
    'Good quality':                  'Good quality matches',
    'Good quality matches':          'Good quality matches',
    'Better matches':                'Good quality matches',
    'Matches':                       'Good quality matches',
    'Good matches':                  'Good quality matches',
    'Bad matches':                   'Bad quality matches',
    'Bad quality matches':           'Bad quality matches',
    'Bad suggestions':               'Bad quality matches',
    'Irrelevant matches':            'Bad quality matches',
    'Irrelevnat matches':            'Bad quality matches',
    'Clingy':                        'Bad quality matches',
    'Weird':                         'Bad quality matches',
    'Boring':                        'Bad quality matches',
    'Ghosting':                      'Ghosting',
    'Flaking':                       'Ghosting',
    'Slow replies':                  'Low effort interactions',
    'LAck of responses':             'Low effort interactions',
    'Low effort':                    'Low effort interactions',
    'Low effort interactions':       'Low effort interactions',
    'Lack of effort':                'Low effort interactions',
    'Hookup culture':                'Hookup culture',
    'Not genuine':                   'Hookup culture',
    'Superficial':                   'Hookup culture',
    'Superficial interactions':      'Hookup culture',
    'Relationship success':          'Relationship success',
    'Relationship success stories':  'Relationship success',
    'Genuine connections':           'Relationship success',
    'Genuine connection':            'Relationship success',
    'Genuine':                       'Relationship success',
    'Relationship mismatch':         'Relationship mismatch',
    'Mismatch':                      'Relationship mismatch',
    'Dont like women first':         'Relationship mismatch',
    'Validation seeking':            'Validation seeking',
    'No matches':                    'No matches',
    'Lack of matches':               'No matches',
    'Lack of matches, conversation': 'No matches',
    'Lack of matches conversation':  'No matches',
    'Low matches':                   'No matches',
    'No connections':                'No matches',
    'Too many options':              'Too many options',
    'Too many likes':                'Too many options',

    # DIMENSION 2 — Emotional Experience & Dating Fatigue
    'Dating fatigue':                'Dating fatigue',
    'Burnout':                       'Dating fatigue',
    'Exhausting':                    'Dating fatigue',
    'Draining':                      'Dating fatigue',
    'Overwhelming':                  'Dating fatigue',
    'Repetitive':                    'Dating fatigue',
    'Repetiitve':                    'Dating fatigue',
    'Hopelessness':                  'Hopelessness',
    'Pointless':                     'Hopelessness',
    'Gave up':                       'Hopelessness',
    'Not going anywhere':            'Hopelessness',
    'Struggling to make connections':'Hopelessness',
    'Difficulty finding someone':    'Hopelessness',
    'Frustrating':                   'Frustration',
    'Upsetting':                     'Frustration',
    'Difficult':                     'Frustration',
    'Convenient':                    'Positive emotional experience',
    'Convenent':                     'Positive emotional experience',
    'Insecurities':                  'Insecurity',
    'Desperation':                   'Insecurity',

    # DIMENSION 3 — Trust, Safety & Authenticity
    'Bots':                          'Fake profiles & bots',
    'Bots/scams':                    'Fake profiles & bots',
    'Fake profiles':                 'Fake profiles & bots',
    'Fake profiles/ scams':          'Fake profiles & bots',
    'Spam':                          'Fake profiles & bots',
    'Anti-bots':                     'Fake profiles & bots',
    'Lack of bots':                  'Fake profiles & bots',
    'Scam':                          'Scams',
    'Scams':                         'Scams',
    'Creeps':                        'Harassment & safety',
    'Rude':                          'Harassment & safety',
    'Unsafe':                        'Harassment & safety',
    'Safety':                        'Harassment & safety',
    'Verification':                  'Verification',
    'Legitimate':                    'Verification',
    'Good IT support':               'Good support',
    'Lack of support':               'Poor support',
    'Lack of supprot':               'Poor support',
    'Reporting issues':              'Poor support',
    'Account ban':                   'Account issues',
    'Account issue':                 'Account issues',
    'Account issues':                'Account issues',

    # DIMENSION 4 — Monetisation & Value Perception
    'Forced subscription':           'Forced subscription',
    'Forced subscriptions':          'Forced subscription',
    'Forced subscritpion':           'Forced subscription',
    'Forced susbcription':           'Forced subscription',
    'Paywall':                       'Forced subscription',
    'Money waste':                   'Poor value for money',
    'Money':                         'Poor value for money',
    'Cost':                          'Poor value for money',
    'No value for money':            'Poor value for money',
    'Useless premium':               'Poor value for money',
    'Pointless premium':             'Poor value for money',
    'Premium feels worthwhile':      'Good value for money',
    'Good premium':                  'Good value for money',
    'Not expensive':                 'Good value for money',
    'Promotions':                    'Good value for money',
    'Price increase':                'Pricing issues',
    'Payment issues':                'Pricing issues',
    'Refund issues':                 'Pricing issues',
    'Cancellation issues':           'Pricing issues',
    'Monetisation manipulation':     'Monetisation manipulation',
    'New profile priority':          'Monetisation manipulation',
    'Disappearing likes':            'Monetisation manipulation',
    'Limited likes':                 'Monetisation manipulation',

    # DIMENSION 5 — Brand Positioning & Competitive Differentiation
    'Better competiion':             'Better than competition',
    'Better competiiton':            'Better than competition',
    'Better competition':            'Better than competition',
    'Better than competition':       'Better than competition',
    'Better competiton':             'Better than competition',
    'Better competiiion':            'Better than competition',
    'Better competiiion':            'Better than competition',
    'Hinge better for serious':      'Worse than competition',
    'Like Tinder':                   'Worse than competition',
    'Like tinder':                   'Worse than competition',
    'No unique':                     'Worse than competition',
    'Not unique':                    'Worse than competition',
    'Compeititon':                   'Worse than competition',
    'Competition':                   'Worse than competition',
    'Women-first good':              'Women-first positive',
    'Women-first bad':               'Women-first negative',

    # DIMENSION 6 — Product & UX Experience
    'UX issue':                      'UX issues',
    'UX issues':                     'UX issues',
    'UX problems':                   'UX issues',
    'UX is slow':                    'UX issues',
    'Decreasing user friendliness':  'UX issues',
    'Poor design':                   'UX issues',
    'Updates needed':                'UX issues',
    'UX improvement':                'UX positive',
    'Good design':                   'UX positive',
    'Algorithm problem':             'Algorithm issues',
    'Algortihm problem':             'Algorithm issues',
    'Changed features':              'Algorithm issues',
    'Doesnt show profile':           'Algorithm issues',
    'Restrictive communication':     'Algorithm issues',
    'Bug':                           'Bugs',
    'Bugs':                          'Bugs',
}

DIMENSION_MAP = {
    'Good quality matches':          'Dim 1 - Match Quality',
    'Bad quality matches':           'Dim 1 - Match Quality',
    'Ghosting':                      'Dim 1 - Match Quality',
    'Low effort interactions':       'Dim 1 - Match Quality',
    'Hookup culture':                'Dim 1 - Match Quality',
    'Relationship success':          'Dim 1 - Match Quality',
    'Relationship mismatch':         'Dim 1 - Match Quality',
    'Validation seeking':            'Dim 1 - Match Quality',
    'No matches':                    'Dim 1 - Match Quality',
    'Too many options':              'Dim 1 - Match Quality',
    'Dating fatigue':                'Dim 2 - Emotional Experience',
    'Hopelessness':                  'Dim 2 - Emotional Experience',
    'Frustration':                   'Dim 2 - Emotional Experience',
    'Positive emotional experience': 'Dim 2 - Emotional Experience',
    'Insecurity':                    'Dim 2 - Emotional Experience',
    'Fake profiles & bots':          'Dim 3 - Trust & Safety',
    'Scams':                         'Dim 3 - Trust & Safety',
    'Harassment & safety':           'Dim 3 - Trust & Safety',
    'Verification':                  'Dim 3 - Trust & Safety',
    'Good support':                  'Dim 3 - Trust & Safety',
    'Poor support':                  'Dim 3 - Trust & Safety',
    'Account issues':                'Dim 3 - Trust & Safety',
    'Forced subscription':           'Dim 4 - Monetisation',
    'Poor value for money':          'Dim 4 - Monetisation',
    'Good value for money':          'Dim 4 - Monetisation',
    'Pricing issues':                'Dim 4 - Monetisation',
    'Monetisation manipulation':     'Dim 4 - Monetisation',
    'Better than competition':       'Dim 5 - Brand & Competition',
    'Worse than competition':        'Dim 5 - Brand & Competition',
    'Women-first positive':          'Dim 5 - Brand & Competition',
    'Women-first negative':          'Dim 5 - Brand & Competition',
    'UX issues':                     'Dim 6 - Product & UX',
    'UX positive':                   'Dim 6 - Product & UX',
    'Algorithm issues':              'Dim 6 - Product & UX',
    'Bugs':                          'Dim 6 - Product & UX',
}

print(f'Category map: {len(CATEGORY_MAP)} raw labels')
print(f'Dimension map: {len(DIMENSION_MAP)} canonical categories')

Category map: 138 raw labels
Dimension map: 35 canonical categories


In [ ]:
def standardise_category(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return None
    raw = str(raw).strip()
    result = CATEGORY_MAP.get(raw, None)
    if result is None:
        print(f'  UNMAPPED: "{raw}" — add to CATEGORY_MAP')
    return result

analyzer = SentimentIntensityAnalyzer()

def score_vader(text):
    if not isinstance(text, str):
        return pd.Series([None, None, None, None, None])
    scores   = analyzer.polarity_scores(text)
    compound = scores['compound']
    label    = 'positive' if compound >= 0.05 else 'negative' if compound <= -0.05 else 'neutral'
    return pd.Series([round(compound,4), round(scores['pos'],4),
                      round(scores['neg'],4), round(scores['neu'],4), label])

print('Helper functions ready.')

Helper functions ready.


In [ ]:
# ---------------------------------------------------------------
# SHEET 1 — Reddit labels
# ---------------------------------------------------------------
df1 = pd.read_csv('Sheet 1-Table 1.csv')
print(f'Sheet 1 loaded: {len(df1)} rows')
print('Standardising categories...')

df1['category_1'] = df1['Category 1'].apply(standardise_category)
df1['category_2'] = df1['Category 2'].apply(standardise_category)
df1['category_3'] = None
df1['dimension_1'] = df1['category_1'].map(DIMENSION_MAP)
df1['dimension_2'] = df1['category_2'].map(DIMENSION_MAP)

sheet1 = pd.DataFrame({
    'source':          df1['source'],
    'original_date':   df1['original_date'],
    'comment_date':    df1['comment_date'],
    'text':            df1['text'],
    'subreddit':       df1['subreddit'],
    'post_title':      df1['post_title'],
    'post_score':      df1['post_score'],
    'comment_score':   df1['comment_score'],
    'rating':          df1['rating'],
    'thumbs_up':       df1['thumbs_up'],
    'vader_compound':  df1['vader_compound'],
    'vader_positive':  df1['vader_positive'],
    'vader_negative':  df1['vader_negative'],
    'vader_neutral':   df1['vader_neutral'],
    'sentiment_label': df1['sentiment_label'],
    'category_1':      df1['category_1'],
    'category_2':      df1['category_2'],
    'category_3':      df1['category_3'],
    'dimension_1':     df1['dimension_1'],
    'dimension_2':     df1['dimension_2'],
})

print(f'Sheet 1 done: {len(sheet1)} rows')

Sheet 1 loaded: 150 rows
Standardising categories...
Sheet 1 done: 150 rows


In [ ]:
import os
print(os.listdir('.'))

['.config', 'Sheet 1-Table 1.csv', 'Sheet 2-Table 1.csv', 'sample_data']


In [ ]:
# ---------------------------------------------------------------
# SHEET 2 — Google Play / Kaggle labels
# ---------------------------------------------------------------
df2 = pd.read_csv('Sheet 2-Table 1.csv')
print(f'Sheet 2 loaded: {len(df2)} rows')
print('Standardising categories and scoring VADER...')

df2['category_1'] = df2['Category 1'].apply(standardise_category)
df2['category_2'] = df2['Category 2'].apply(standardise_category)
df2['category_3'] = df2['Category 3'].apply(standardise_category)
df2['dimension_1'] = df2['category_1'].map(DIMENSION_MAP)
df2['dimension_2'] = df2['category_2'].map(DIMENSION_MAP)
df2['original_date'] = pd.to_datetime(df2['at'], errors='coerce').dt.strftime('%Y-%m-%d')

# Score VADER on Google Play review text
vader_scores = df2['content'].apply(score_vader)
vader_scores.columns = ['vader_compound','vader_positive','vader_negative','vader_neutral','sentiment_label']

sheet2 = pd.DataFrame({
    'source':          'google_play',
    'original_date':   df2['original_date'],
    'comment_date':    None,
    'text':            df2['content'],
    'subreddit':       None,
    'post_title':      None,
    'post_score':      None,
    'comment_score':   None,
    'rating':          df2['score'],
    'thumbs_up':       df2['thumbsUpCount'],
    'vader_compound':  vader_scores['vader_compound'].values,
    'vader_positive':  vader_scores['vader_positive'].values,
    'vader_negative':  vader_scores['vader_negative'].values,
    'vader_neutral':   vader_scores['vader_neutral'].values,
    'sentiment_label': vader_scores['sentiment_label'].values,
    'category_1':      df2['category_1'],
    'category_2':      df2['category_2'],
    'category_3':      df2['category_3'],
    'dimension_1':     df2['dimension_1'],
    'dimension_2':     df2['dimension_2'],
})

print(f'Sheet 2 done: {len(sheet2)} rows')

Sheet 2 loaded: 133 rows
Standardising categories and scoring VADER...
Sheet 2 done: 133 rows


In [ ]:
# ---------------------------------------------------------------
# COMBINE & SUMMARISE
# ---------------------------------------------------------------
training = pd.concat([sheet1, sheet2], ignore_index=True)
training['original_date'] = pd.to_datetime(training['original_date'], errors='coerce')
training = training.sort_values('original_date', ascending=False).reset_index(drop=True)

print(f'Training set: {len(training)} rows total')
print(f'\nBy source:')
print(training['source'].value_counts().to_string())
print(f'\nTop 15 categories:')
print(training['category_1'].value_counts().head(15).to_string())
print(f'\nDimension distribution:')
print(training['dimension_1'].value_counts().to_string())
print(f'\nUnmapped rows: {training["category_1"].isna().sum()}')

Training set: 283 rows total

By source:
source
google_play    169
reddit         108
app_store        6

Top 15 categories:
category_1
Forced subscription        33
No matches                 26
Bad quality matches        17
Account issues             14
Ghosting                   14
Dating fatigue             14
Poor value for money       14
UX issues                  12
Pricing issues             11
Hookup culture             11
Relationship mismatch      11
Low effort interactions    11
Poor support                9
Good quality matches        8
Fake profiles & bots        8

Dimension distribution:
dimension_1
Dim 1 - Match Quality           104
Dim 4 - Monetisation             64
Dim 3 - Trust & Safety           47
Dim 2 - Emotional Experience     27
Dim 6 - Product & UX             25
Dim 5 - Brand & Competition      16

Unmapped rows: 0


/tmp/ipykernel_3555/2902180870.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  training = pd.concat([sheet1, sheet2], ignore_index=True)


In [ ]:
training.to_csv('bumble_training_set.csv', index=False)
print(f'Saved {len(training)} rows to bumble_training_set.csv')
print(f'Columns: {list(training.columns)}')

Saved 283 rows to bumble_training_set.csv
Columns: ['source', 'original_date', 'comment_date', 'text', 'subreddit', 'post_title', 'post_score', 'comment_score', 'rating', 'thumbs_up', 'vader_compound', 'vader_positive', 'vader_negative', 'vader_neutral', 'sentiment_label', 'category_1', 'category_2', 'category_3', 'dimension_1', 'dimension_2']


In [ ]:
try:
    from google.colab import files
    files.download('bumble_training_set.csv')
    print('Download triggered.')
except ImportError:
    print('Not in Colab — file saved locally.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.
